In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install scikit-multilearn
!pip install librosa

import os
import shutil
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from skmultilearn.model_selection import iterative_train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report

# ==========================================
# CONFIGURACIÓN GLOBAL
# ==========================================
config = {
    # Rutas
    "ruta_zip_drive": '/content/drive/MyDrive/TFM/espectrogramas_tfm.zip',
    "ruta_csv_drive": '/content/drive/MyDrive/TFM/metadatos_espectrogramas.csv',
    "ruta_checkpoint": '/content/drive/MyDrive/TFM/checkpoints/MOBILENET',
    "ruta_resultados": '/content/drive/MyDrive/TFM/resultados/MOBILENET',
    "ruta_local_espectrogramas": '/content/espectrogramas_tfm',
    
    # Hiperparámetros de Entrenamiento
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "epochs": 50,
    "paciencia_maxima": 5,
    "umbral_clasificacion": 0.5,
    
    # Metadatos del Dominio
    "emociones": [
        'calm', 'dark', 'emotional', 'energetic', 'epic', 'happy',
        'melancholic', 'powerful', 'relaxing', 'romantic', 'sad', 'upbeat'
    ]
}

# Creación de carpetas y rutas derivadas
os.makedirs(config["ruta_checkpoint"], exist_ok=True)
os.makedirs(config["ruta_resultados"], exist_ok=True)
config["ruta_csv_log"] = os.path.join(config["ruta_checkpoint"], 'historial_metricas.csv')
config["ruta_latest"] = os.path.join(config["ruta_checkpoint"], 'latest_checkpoint.pth')
config["ruta_best"] = os.path.join(config["ruta_checkpoint"], 'best_model.pth')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Configuración cargada. Ejecutando en: {device}")

In [ ]:
# ==========================================
# GESTIÓN DE DATOS Y DATALOADERS
# ==========================================
class JamendoDataset(Dataset):
    """Dataset personalizado para cargar espectrogramas bajo demanda."""
    def __init__(self, x_files, y_labels, folder_path):
        self.x_files = x_files
        self.y_labels = y_labels
        self.folder_path = folder_path

    def __len__(self):
        return len(self.x_files)

    def __getitem__(self, idx):
        file_name = self.x_files[idx][0]
        matriz_mel = np.load(os.path.join(self.folder_path, file_name))
        x_tensor = torch.tensor(np.expand_dims(matriz_mel, axis=0), dtype=torch.float32)
        y_tensor = torch.tensor(self.y_labels[idx], dtype=torch.float32)
        return x_tensor, y_tensor


def prepare_local_data(cfg):
    """Descomprime el dataset en el almacenamiento local de Colab si no existe."""
    if not os.path.exists(cfg["ruta_local_espectrogramas"]):
        print("Copiando y descomprimiendo el ZIP de espectrogramas...")
        shutil.copy2(cfg["ruta_zip_drive"], '/content/espectrogramas_tfm.zip')
        os.system("unzip -q /content/espectrogramas_tfm.zip -d /content/")
        os.remove('/content/espectrogramas_tfm.zip')
        print("Espectrogramas listos en local.")


def get_dataloaders(cfg):
    """Genera los DataLoaders estratificados para Train, Val y Test."""
    prepare_local_data(cfg)
    
    df = pd.read_csv(cfg["ruta_csv_drive"])
    X = df[['CHUNK_FILE']].values
    y = df[cfg["emociones"]].values

    print("\nAplicando Iterative Stratification para Multi-Label...")
    X_train_val, y_train_val, X_test, y_test = iterative_train_test_split(X, y, test_size=0.20)
    X_train, y_train, X_val, y_val = iterative_train_test_split(X_train_val, y_train_val, test_size=0.125)

    print(f"Entrenamiento (Train): {len(X_train)} ventanas")
    print(f"Validación (Val):      {len(X_val)} ventanas")
    print(f"Evaluación (Test):     {len(X_test)} ventanas\n")

    train_dataset = JamendoDataset(X_train, y_train, cfg["ruta_local_espectrogramas"])
    val_dataset = JamendoDataset(X_val, y_val, cfg["ruta_local_espectrogramas"])
    test_dataset = JamendoDataset(X_test, y_test, cfg["ruta_local_espectrogramas"])

    train_loader = DataLoader(train_dataset, batch_size=cfg["batch_size"], shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# ==========================================
# 2. ARQUITECTURA (MobileNet) ADAPTADA (Sin Autoatención)
# ==========================================
class MusicMobileAttention(nn.Module):
    def __init__(self, num_classes=12):
        super(MusicMobileAttention, self).__init__()

        # --- FASE 1: VISIÓN ARTIFICIAL LIGERA (MobileNetV3) ---
        mobilenet = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

        original_conv = mobilenet.features[0][0]
        mobilenet.features[0][0] = nn.Conv2d(1, original_conv.out_channels,
                                             kernel_size=original_conv.kernel_size,
                                             stride=original_conv.stride,
                                             padding=original_conv.padding,
                                             bias=False)

        self.feature_extractor = mobilenet.features

        # --- FASE 1.5: EL CUELLO DE BOTELLA Y EL RESHAPE ---
        self.freq_pool = nn.AdaptiveAvgPool2d((4, None))

        d_model_cnn_out = 576 * 4
        self.d_model = 512

        self.bottleneck = nn.Linear(d_model_cnn_out, self.d_model)

        # --- FASE 4: CLASIFICADOR ---
        self.fc = nn.Linear(self.d_model, num_classes)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc.weight)
        if self.fc.bias is not None:
            nn.init.zeros_(self.fc.bias)
        nn.init.xavier_uniform_(self.bottleneck.weight)
        if self.bottleneck.bias is not None:
            nn.init.zeros_(self.bottleneck.bias)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.freq_pool(x)
        batch_size, channels, freq, time = x.size()
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(batch_size, time, channels * freq)
        x = self.bottleneck(x)
        x = torch.relu(x)
        
        # Colapso del eje temporal (promedio) en lugar de usar Transformer
        x = torch.mean(x, dim=1)
        
        return self.fc(x)


In [ ]:
# ==========================================
# RUTINAS DE ENTRENAMIENTO Y VALIDACIÓN
# ==========================================
def train_one_epoch(model, loader, criterion, optimizer, epoch, cfg):
    """Entrena el modelo por una sola época y devuelve la pérdida media."""
    model.train()
    total_loss = 0.0
    loop = tqdm(loader, desc=f"Epoch {epoch+1}/{cfg['epochs']} [Train]")
    
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        loop.set_postfix(loss=loss.item())
        
    return total_loss / len(loader.dataset)


def validate_one_epoch(model, loader, criterion):
    """Evalúa el modelo en el conjunto de validación."""
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            
    return total_loss / len(loader.dataset)


def train_model(model, train_loader, val_loader, cfg):
    """Bucle principal de entrenamiento con Early Stopping y guardado de checkpoints."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW([
        {'params': model.feature_extractor.parameters(), 'lr': 1e-5},
        {'params': model.bottleneck.parameters()},
        {'params': model.fc.parameters()}
    ], lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

    start_epoch = 0
    best_val_loss = float('inf')
    paciencia_count = 0

    # Inicializar log
    if not os.path.exists(cfg["ruta_csv_log"]):
        with open(cfg["ruta_csv_log"], mode='w', newline='') as f:
            csv.writer(f).writerow(['Epoch', 'Train_Loss', 'Val_Loss'])

    # Reanudación Automática
    if os.path.exists(cfg["ruta_latest"]):
        print(f"Reanudando desde checkpoint previo: {cfg['ruta_latest']}")
        checkpoint = torch.load(cfg["ruta_latest"], map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint.get('best_val_loss', float('inf'))

    for epoch in range(start_epoch, cfg["epochs"]):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, epoch, cfg)
        val_loss = validate_one_epoch(model, val_loader, criterion)
        
        print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        scheduler.step(val_loss)

        # Loggeamos métricas
        with open(cfg["ruta_csv_log"], mode='a', newline='') as f:
            csv.writer(f).writerow([epoch+1, train_loss, val_loss])

        estado_actual = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss
        }
        torch.save(estado_actual, cfg["ruta_latest"])

        # Control de Early Stopping
        if val_loss < best_val_loss:
            print("¡Mejora! Guardando best_model.pth...")
            best_val_loss = val_loss
            estado_actual['best_val_loss'] = best_val_loss
            torch.save(estado_actual, cfg["ruta_best"])
            paciencia_count = 0
        else:
            paciencia_count += 1
            print(f"Sin mejora ({paciencia_count}/{cfg['paciencia_maxima']})")

        if paciencia_count >= cfg["paciencia_maxima"]:
            print("Early Stopping detonado. Entrenamiento finalizado.")
            break

In [ ]:
# ==========================================
# RUTINAS DE EVALUACIÓN Y VISUALIZACIÓN
# ==========================================
def evaluate_model(model, test_loader, cfg):
    """Realiza inferencia sobre el test ciego y muestra reporte de métricas globales e individuales."""
    print("\nINICIANDO EVALUACIÓN SOBRE CONJUNTO DE TEST CIEGO...")
    
    # Cargar los mejores pesos encontrados
    checkpoint = torch.load(cfg["ruta_best"], map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluando"):
            inputs = inputs.to(device)
            probs = torch.sigmoid(model(inputs))
            all_preds.extend(probs.cpu().numpy())
            all_targets.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    # Cálculo Global
    roc_auc_macro = roc_auc_score(all_targets, all_preds, average='macro')
    pr_auc_macro = average_precision_score(all_targets, all_preds, average='macro')

    print(f"\nMÉTRICAS GLOBALES INDEPENDIENTES DEL UMBRAL:")
    print(f"ROC-AUC (Macro): {roc_auc_macro:.4f}")
    print(f"PR-AUC (Macro):  {pr_auc_macro:.4f}\n")

    # Reporte dependiente de umbral
    preds_binarias = (all_preds > cfg["umbral_clasificacion"]).astype(int)
    print(f"REPORTE DE CLASIFICACIÓN (Umbral {cfg['umbral_clasificacion']}):")
    print(classification_report(all_targets, preds_binarias, target_names=cfg["emociones"], zero_division=0))
    
    return all_preds, all_targets


def plot_metrics(all_preds, all_targets, cfg):
    """Genera y guarda las curvas de aprendizaje y gráficos de métricas AUC."""
    # 1. Curva de Aprendizaje
    try:
        df_log = pd.read_csv(cfg["ruta_csv_log"])
        checkpoint_best = torch.load(cfg["ruta_best"], map_location=device)
        best_epoch = checkpoint_best['epoch'] + 1
        
        plt.figure(figsize=(10, 6))
        plt.plot(df_log['Epoch'], df_log['Train_Loss'], label='Train Loss', color='blue')
        plt.plot(df_log['Epoch'], df_log['Val_Loss'], label='Validation Loss', color='orange')
        plt.axvline(x=best_epoch, color='red', linestyle='--', label='Mejor Época')
        plt.title('Curva de Aprendizaje (Loss)')
        plt.xlabel('Época')
        plt.ylabel('BCE Loss')
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(cfg["ruta_resultados"], 'curva_aprendizaje.png'), dpi=300)
        plt.show()
    except Exception as e:
        print(f"Aviso: No se pudo dibujar la curva de aprendizaje ({e})")

    # 2. Gráfico de barras ROC-AUC
    roc_aucs_por_clase = []
    for i in range(len(cfg["emociones"])):
        try:
            score = roc_auc_score(all_targets[:, i], all_preds[:, i])
        except ValueError:
            score = 0.0
        roc_aucs_por_clase.append(score)

    plt.figure(figsize=(12, 6))
    bars = plt.bar(cfg["emociones"], roc_aucs_por_clase, color='teal')
    plt.axhline(y=0.5, color='red', linestyle='--', label='Aleatoriedad (0.5)')
    plt.title('Rendimiento ROC-AUC por Emoción')
    plt.ylabel('ROC-AUC Score')
    plt.ylim(0, 1.05)
    plt.xticks(rotation=45)
    plt.legend()
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f'{yval:.2f}', ha='center', va='bottom', fontsize=9)
        
    plt.tight_layout()
    plt.savefig(os.path.join(cfg["ruta_resultados"], 'roc_auc_por_clase.png'), dpi=300)
    plt.show()
    
    print("\nGráficas guardadas en Google Drive.")

In [ ]:
# ==========================================
# PUNTO DE ENTRADA PRINCIPAL
# ==========================================
if __name__ == "__main__":
    # 1. Cargar Datos
    train_loader, val_loader, test_loader = get_dataloaders(config)
    
    # 2. Inicializar Modelo
    model = MusicMobileAttention(num_classes=len(config["emociones"])).to(device)
    
    # 3. Entrenamiento Automatizado
    train_model(model, train_loader, val_loader, config)
    
    # 4. Evaluación del Estado Final
    preds, targets = evaluate_model(model, test_loader, config)
    plot_metrics(preds, targets, config)